# Notebook 01 — Exploratory Data Analysis (EDA)

## What is EDA?
Before training any model, we first LOOK at our data.
We want to answer these simple questions:
- How many images do we have per class?
- What do the images look like?
- Are the classes balanced or imbalanced?
- What size are the images?

This notebook generates Figure 1 and Figure 2 for your report.

---
##  Import Libraries

We need these tools:
- `os` / `pathlib` → to work with folders and files
- `PIL` (Pillow)   → to open and read images
- `numpy`          → for number calculations (mean, min, max)
- `matplotlib`     → to draw charts and display images

In [ ]:
import os
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path
from PIL import Image

print('All libraries imported successfully!')

print('All libraries imported successfully!')

: 

---
## Set/Check the Path to Your Data

We tell Python WHERE our images are saved.

Your folder structure should look like this:
```
data/
  raw/
    benign/       437 images
    malignant/    210 images
    normal/       133 images
```

In [ ]:
# Path to your raw images folder
DATA_FOLDER = Path('data/raw')

# The three classes in our dataset
CLASSES = ['benign', 'malignant', 'normal']

# A colour for each class (used in charts later)
# blue = benign   red = malignant   green = normal
COLORS = ['#2196F3', '#F44336', '#4CAF50']

# Check that all three folders actually exist
for cls in CLASSES:
    folder = DATA_FOLDER / cls
    if folder.exists():
        print(f'FOUND  ->  {folder}')
    else:
        print(f'NOT FOUND  ->  {folder}   check your path!')

---
##  Count How Many Images Are in Each Class

We go into each folder and count the .png files.

Expected result:
- benign    = 437
- malignant = 210
- normal    = 133
- TOTAL     = 780

In [ ]:
# We store the counts here
# After this cell: class_counts = {'benign': 437, 'malignant': 210, 'normal': 133}
class_counts = {}

print('Image count per class:')
print('-' * 30)

for cls in CLASSES:

    # Get a list of all .png files in this class folder
    all_images = list((DATA_FOLDER / cls).glob('*.png'))

    # Count how many files are in the list
    count = len(all_images)

    # Save the result into the dictionary
    class_counts[cls] = count

    print(f'  {cls:<12} :  {count} images')

print('-' * 30)
print(f'  TOTAL        :  {sum(class_counts.values())} images')

---
##  Check Image Sizes

Ultrasound images come in different sizes depending on the scanner used.

We need to know this because we will resize everything to 224x224 pixels
later. ResNet-18 (our model) requires exactly 224x224 as input.

In [ ]:
print('Image size check per class:')
print('-' * 60)

for cls in CLASSES:

    all_images = list((DATA_FOLDER / cls).glob('*.png'))

    widths  = []
    heights = []

    # Open each image and record its width and height
    for img_path in all_images:
        img    = Image.open(img_path)
        w, h   = img.size       # img.size gives us (width, height)
        widths.append(w)
        heights.append(h)

    print(f'Class: {cls}')
    print(f'  Width   smallest={min(widths)}   largest={max(widths)}   average={int(np.mean(widths))}')
    print(f'  Height  smallest={min(heights)}   largest={max(heights)}   average={int(np.mean(heights))}')
    print()

---
## FIGURE 1: Show Sample Images from Each Class

We display 4 sample images from each class in a grid.
This becomes Figure 1 in your report.

The grid looks like this:
```
           Sample 1    Sample 2    Sample 3    Sample 4
Benign     [image]     [image]     [image]     [image]
Malignant  [image]     [image]     [image]     [image]
Normal     [image]     [image]     [image]     [image]
```

In [ ]:
# Make sure the output folder exists
os.makedirs('outputs/figures', exist_ok=True)

# Create the figure: 3 rows, 4 columns
fig, axes = plt.subplots(3, 4, figsize=(14, 10))

# Main title at the top of the whole figure
fig.suptitle('Figure 1: Sample Breast Ultrasound Images per Class',
             fontsize=14, fontweight='bold')

# Loop: each class = one row
for row_index, cls in enumerate(CLASSES):

    # Get the first 4 images from this class folder
    sample_images = sorted((DATA_FOLDER / cls).glob('*.png'))[:4]

    # Loop: each image = one column
    for col_index, img_path in enumerate(sample_images):

        # Open image in grayscale
        # convert('L') = convert to grayscale (L stands for Luminance)
        img = Image.open(img_path).convert('L')

        # Display the image in the correct grid cell
        axes[row_index, col_index].imshow(img, cmap='gray')

        # Turn off axis ticks (not useful for image display)
        axes[row_index, col_index].axis('off')

        # Add column title on the FIRST ROW only
        if row_index == 0:
            axes[row_index, col_index].set_title(f'Sample {col_index + 1}', fontsize=10)

        # Add class name label on the LEFT SIDE of the first column only
        if col_index == 0:
            axes[row_index, col_index].set_ylabel(
                cls.capitalize(),
                fontsize=12,
                fontweight='bold',
                color=COLORS[row_index]
            )

plt.tight_layout()
plt.savefig('outputs/figures/fig1_sample_images.png', dpi=150, bbox_inches='tight')
plt.show()

print('Figure 1 saved  ->  outputs/figures/fig1_sample_images.png')

---
## FIGURE 2: Class Distribution Chart

We show how many images are in each class using two charts:
- Left  → Bar chart showing exact counts
- Right → Pie chart showing percentages

This becomes Figure 2 in your report.

Why does this matter?
If one class has far more images than others, the model will be BIASED.
It will get good at predicting the big class and bad at the small ones.
We must fix this in training using WeightedRandomSampler.

In [ ]:
# Get counts as a list: [437, 210, 133]
counts = [class_counts[cls] for cls in CLASSES]
total  = sum(counts)

# Create figure with 2 side-by-side charts
fig, axes = plt.subplots(1, 2, figsize=(12, 5))
fig.suptitle('Figure 2: Class Distribution in the Dataset',
             fontsize=13, fontweight='bold')


# ── LEFT: Bar chart ──────────────────────────────────────────

bars = axes[0].bar(
    CLASSES,       # x-axis: class names
    counts,        # y-axis: number of images
    color=COLORS,
    edgecolor='black',
    linewidth=0.8
)

# Put the number on top of each bar
for bar, count in zip(bars, counts):
    axes[0].text(
        bar.get_x() + bar.get_width() / 2,  # x = centre of bar
        bar.get_height() + 8,               # y = just above bar
        str(count),
        ha='center',
        fontweight='bold',
        fontsize=11
    )

axes[0].set_title('Image Count per Class')
axes[0].set_ylabel('Number of Images')
axes[0].set_xlabel('Class')
axes[0].set_ylim(0, max(counts) + 70)  # extra space so numbers are not cut off


# ── RIGHT: Pie chart ─────────────────────────────────────────

axes[1].pie(
    counts,
    labels=[c.capitalize() for c in CLASSES],
    colors=COLORS,
    autopct='%1.1f%%',           # show percentage inside each slice
    startangle=140,
    wedgeprops={'edgecolor': 'white', 'linewidth': 2}
)
axes[1].set_title('Class Proportion (%)')


plt.tight_layout()
plt.savefig('outputs/figures/fig2_class_distribution.png', dpi=150, bbox_inches='tight')
plt.show()

print('Figure 2 saved  ->  outputs/figures/fig2_class_distribution.png')
print()
print('Class imbalance summary:')
for cls, count in zip(CLASSES, counts):
    print(f'  {cls:<12} :  {count} images  ({count/total*100:.1f}%)')
print()
print('Benign is 56% of the data  ->  class imbalance must be handled in training!')

---
## Final Summary

This prints a clean summary of everything we found.
Use these numbers in your report Methods section.

In [ ]:
print('=' * 50)
print('  EDA SUMMARY')
print('=' * 50)
print(f'  Total images       :  {sum(class_counts.values())}')
print(f'  Benign             :  {class_counts["benign"]}   (56.0%)')
print(f'  Malignant          :  {class_counts["malignant"]}   (26.9%)')
print(f'  Normal             :  {class_counts["normal"]}   (17.1%)')
print(f'  Image format       :  PNG')
print(f'  Image colour       :  Grayscale')
print(f'  Average image size :  ~500 x 500 pixels')
print(f'  Resize target      :  224 x 224 pixels (required by ResNet-18)')
print(f'  Mask files         :  Removed manually before analysis')
print(f'  Class imbalance    :  YES  ->  will use WeightedRandomSampler')
print('=' * 50)
print()
print('Figures saved for report:')
print('  outputs/figures/fig1_sample_images.png       ->  Figure 1')
print('  outputs/figures/fig2_class_distribution.png  ->  Figure 2')
print()
print('NEXT STEP:  Run  02_Preprocessing.ipynb')
print('That notebook creates your train / val / test split.')